# 04 — Final Results (report-ready)

Compiles the whole project into the figures and tables that go in the report:

1. the **ablation** verdict — does behaviour beat quant-only?
2. a realistic **backtest** equity curve vs buy-and-hold,
3. **SHAP** interpretability — did fear/greed/herd actually drive the calls?

All logic lives in `src/`; this notebook orchestrates and renders. Figures are saved to
`results/figures/`.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd, matplotlib.pyplot as plt

from src.config import load_config
from src.experiments import run_ablation, build_model_frame, feature_columns
from src.data.loader import load_ohlcv
from src.data.splitter import single_split
from src.models import build_model
from src.backtest import run_backtest, buy_and_hold, signal_from_predictions
from src.evaluation import financial_metrics
from src.visualize import (plot_ablation_comparison, plot_equity_curve, plot_drawdown,
                           plot_confusion, plot_feature_importance, shap_summary,
                           behavioral_shap_ranking)

FIG = '../results/figures'
config = load_config('../config.yaml')
config['data']['tickers']

## 1. The ablation verdict

Does adding fear/greed/herd beat the quant-only baseline?

In [ ]:
results = run_ablation(config)
summary, significance = results['summary'], results['significance']
summary

In [ ]:
fig = plot_ablation_comparison(summary, metric='accuracy', save_path=f'{FIG}/ablation_accuracy.png')
fig2 = plot_ablation_comparison(summary, metric='sharpe', save_path=f'{FIG}/ablation_sharpe.png')
plt.show()

**Is the gap statistically real?** (paired test across folds)

In [ ]:
significance

## 2. Realistic backtest

Train the behaviour-aware model on the in-sample window, then trade its out-of-sample
predictions through `src.backtest` (positions, transaction costs) and compare to buy-and-hold.

In [ ]:
ticker = config['data']['tickers'][0]
raw = load_ohlcv(ticker, config['data']['start_date'], config['data']['end_date'], config['data']['raw_dir'])
frame = build_model_frame(raw, config)
cols = feature_columns(frame, 'quant_plus_behavioral')

train_idx, test_idx = single_split(frame, config['split'].get('test_size', 0.2))
model = build_model('xgboost', config).fit(frame.loc[train_idx, cols], frame.loc[train_idx, 'target'])
pred = model.predict(frame.loc[test_idx, cols])

test_prices = frame.loc[test_idx, 'Close']
signals = signal_from_predictions(pred, index=test_idx, allow_short=config['backtest'].get('allow_short', False))
bt = run_backtest(test_prices, signals,
                  initial_capital=config['backtest']['initial_capital'],
                  transaction_cost=config['backtest']['transaction_cost'])
financial_metrics(bt['equity'])

In [ ]:
bench = buy_and_hold(test_prices, config['backtest']['initial_capital'])
plot_equity_curve(bt['equity'], benchmark=bench, save_path=f'{FIG}/{ticker}_equity.png')
plot_drawdown(bt['equity'], save_path=f'{FIG}/{ticker}_drawdown.png')
plot_confusion(frame.loc[test_idx, 'target'], pred, save_path=f'{FIG}/{ticker}_confusion.png')
plt.show()

## 3. SHAP — did behaviour actually drive the decisions?

The payoff question. If fear/greed/herd rank near the top of mean |SHAP|, the model genuinely
leaned on investor psychology — not just price technicals.

In [ ]:
ranking = behavioral_shap_ranking(model, frame.loc[test_idx, cols], max_samples=300)
ranking.head(15)

In [ ]:
shap_summary(model, frame.loc[test_idx, cols], save_path=f'{FIG}/{ticker}_shap_summary.png', max_samples=300)
plt.show()

In [ ]:
imp = model.feature_importances()
plot_feature_importance(imp, top_n=15, save_path=f'{FIG}/{ticker}_importance.png')
plt.show()

## Conclusion

_Fill in from the outputs above:_

- **Prediction:** behavioral features changed accuracy by **… pp** (significance p = …).
- **Trading:** strategy Sharpe **…** vs buy-and-hold **…**; max drawdown **…**.
- **Psychology:** the most influential behavioral signal was **…** (SHAP rank …).

Ties back to the proposal's expected outcomes: a behaviour-aware strategy, a traditional-vs-
behavioral comparison, and insights into the role of behavioral factors.